In [205]:
from pathlib import Path
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.model_selection import train_test_split
from peft import LoraConfig, TaskType
from torch.utils.data import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments, Trainer

random_state = 2026

In [206]:
def generate_dataset(path="/content/diabetes_binary_health_indicators_BRFSS2015.csv"):
  df = pd.read_csv(path)
  rename_cols = {
      "Diabetes_binary": "diabetes",
      "MentHlth": "days_mentally_ill",
      "Smoker": "smoking",
      "HvyAlcoholConsump": "alchohol_consumption",
      "PhysActivity": "physical_activity",
      "Fruits": "consumed_fruits",
      "Veggies": "consumed_vegetables",
      "PhysHlth": "days_physically_ill",
      "DiffWalk": "difficulty_walking",
      "GenHlth": "general_health",
  }
  columns_filter = [col for col in rename_cols.keys()]
  df = df[columns_filter]
  df = df.rename(columns=rename_cols)
  df["diabetes"] = df.diabetes.astype(int)

  df = df.dropna()
  return df



In [207]:
df = generate_dataset()
df

,diabetes,days_mentally_ill,smoking,alchohol_consumption,physical_activity,consumed_fruits,consumed_vegetables,days_physically_ill,difficulty_walking,general_health
0,0,18.0,1.0,0.0,0.0,0.0,1.0,15.0,1.0,5.0
1,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,3.0
2,0,30.0,0.0,0.0,0.0,1.0,0.0,30.0,1.0,5.0
3,0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,2.0
4,0,3.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,2.0
...,...,...,...,...,...,...,...,...,...,...
253675,0,0.0,0.0,0.0,0.0,1.0,1.0,5.0,0.0,3.0
253676,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0
253677,0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
253678,0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,3.0


In [208]:
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
model = AutoModelForSequenceClassification.from_pretrained("dmis-lab/biobert-base-cased-v1.1", num_labels=2, dtype=torch.bfloat16)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

In [209]:
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

0

In [210]:
temp_df = df.head(50)

y = temp_df["diabetes"]
X = temp_df.drop(columns=["diabetes"])

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=random_state, shuffle=False)



In [211]:
def text_format(train_dataset):
  texts = []
  for i in range(len(train_dataset)):
    features = train_dataset.iloc[i]

    consumed_vegetables = "Yes"  if features.consumed_vegetables == 1 else "No"
    consumed_fruits = "Yes"  if features.consumed_fruits == 1 else "No"
    smoking = "Yes" if features.smoking == 1 else "No"
    alcohol = "Yes" if features.alchohol_consumption == 1 else "No"
    difficulty_walking = "Yes" if features.difficulty_walking == 1 else "No"
    physically_active = "Yes" if features.physical_activity == 1 else "No"


    build_up = []
    build_up.append(f"Eats 1 or more vegetable a day: {consumed_vegetables};")
    build_up.append(f"Eats 1 or more fruit a day: {consumed_fruits};")
    build_up.append(f"Number of days spent mentally ill for last 30 days: {features.days_mentally_ill};")
    build_up.append(f"Number of days spent physically ill for last 30 days: {features.days_physically_ill};")
    build_up.append(f"Has been physically active in the last 30 days: {physically_active};")
    build_up.append(f"Has difficulty walking stairs: {difficulty_walking};")
    build_up.append(f"Overall health score from self assessment (1=excellent 2=very good 3=good 4=fair 5=poor): {features.general_health};")
    build_up.append(f"Has smoked more than 100 cigarettes: {smoking};")
    build_up.append(f"Heavy alchohol consumption (men=14 or more drinks a week, women=7 or more drink a week): {alcohol};")

    text = "".join(build_up)

    texts.append(text)
  return texts


In [212]:
class DiabetesDataset(Dataset):

  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels

  def __getitem__(self, idx):
    result = {key: val[idx] for key, val in self.encodings.items()}
    result["labels"] = self.labels[idx]
    return result

  def __len__(self):
    return len(self.labels)

In [213]:
texts = text_format(X_train)


train_encodings = tokenizer(texts, truncation=True, padding=True, return_tensors="pt")


texts = text_format(X_test)


test_encodings = tokenizer(texts, truncation=True, padding=True, return_tensors="pt")



In [214]:
peft_config = LoraConfig(task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=32, lora_dropout=0.1)
model = get_peft_model(model, peft_config)

In [215]:
model.print_trainable_parameters()

trainable params: 296,450 || all params: 108,608,260 || trainable%: 0.2730


In [217]:
training_args = TrainingArguments(
    output_dir="./train_args",
    learning_rate=1e-3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=DiabetesDataset(train_encodings, y_train),
    eval_dataset=DiabetesDataset(test_encodings, y_test)
)


In [218]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyError: 0